# 21 · RAG-Fusion：把 Multi-Query 组装成产品

> RAG-Fusion = 原问题 + LLM 生成多个变体 → 多路并行检索 → **RRF 融合** → Top-K 给 LLM。是 Multi-Query(19) 与 Hybrid 融合(17) 的组合拳。

**本文件覆盖知识点**：RAG-Fusion 全流程 / Multi-query / Parallel Retrieval / RRF / Weighted RRF / Query Weight / Fusion Strategy

```text
Original Query
      │ LLM
      ▼
Generate Multiple Queries
      │
      ▼
Multiple Retrieval (并行)
      │
      ▼
RRF 融合
      │
      ▼
Top-K → LLM → Answer
```

In [ ]:
# .env 配置
from dotenv import load_dotenv; load_dotenv()
import os, json, numpy as np
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

# ---- 复用：多查询生成 ----
def generate_queries(query, n=3):
    from dashscope import Generation
    prompt = f'把问题改写成 {n} 个侧重不同的检索查询，输出 JSON 字符串数组。\n问题: {query}'
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':prompt}],
                        api_key=API_KEY, result_format='message')
    raw = r.output.choices[0].message.content.strip().strip('`')
    if raw.startswith('json'): raw = raw[4:].strip()
    try:
        return [query] + json.loads(raw)
    except Exception:
        return [query]

# ---- 模拟检索器（每个查询返回一个"doc排名"，生产换成真实向量/BM25） ----
corpus = {
    'A': '客服机器人提供免费试用额度',
    'B': '付费套餐分为基础版、专业版、企业版',
    'C': '基础版与专业版差异在并发数与知识库容量',
    'D': '企业版支持私有化部署与专属支持',
    'E': '产品支持 7x24 小时自动应答',
}
def fake_retrieve(q, k=3):
    """按字符重叠做的玩具检索（演示 RRF，非真实检索）"""
    qs = set(''.join(q.split()))
    scored = sorted(corpus.items(), key=lambda kv: -len(qs & set(''.join(kv[1].split()))))
    return [doc for doc, _ in scored[:k]]

# ---- RRF 融合 ----
def rrf_fuse(rankings, k=60):
    score = {}
    for rk in rankings:
        for rank, doc in enumerate(rk):
            score[doc] = score.get(doc, 0) + 1.0 / (k + rank + 1)
    return sorted(score, key=score.get, reverse=True)

def rag_fusion(query, top_k=3):
    queries = generate_queries(query, n=3) if (API_KEY and '你的' not in API_KEY) else [query]
    fused = rrf_fuse([fake_retrieve(q) for q in queries])
    return [(d, corpus[d]) for d in fused[:top_k]]

print('RAG-Fusion 结果:')
for d, t in rag_fusion('星云客服机器人有哪些版本'):
    print(f'  [{d}] {t}')

## 融合策略再升级

| 策略 | 公式/做法 | 何时用 |
|------|-----------|--------|
| RRF | `Σ 1/(k+rank)` | 默认首选 |
| **Weighted RRF** | 给各查询榜单乘权重 `Σ w_q/(k+rank)` | 原问题比改写更重要 |
| Query Weight | 显式调原查询/改写查询的权重 | 有时原句最重要 |
| Score Fusion | 归一化后加权相加 | 想保留“分数强度” |

## 成本与质量

- **收益**：多角度召回，Recall 显著提高；
- **成本**：n 倍查询延迟/费用 → 用**并行**压缩延迟，n 一般 3~5；
- **验收**：仍要回到第 33/34 课评测看 Recall 提升是否值回成本。

## 小结

- RAG-Fusion = **多查询 + 并行检索 + RRF 融合**；
- 融合策略：RRF 起步，Weighted 优化，QoS 看评测；
- 这是“Query Transformation”向“精排前召回优化”的集大成。